# GPT baselines (gpt-4o-mini)

Zero-shot evaluation on the same task families used in:
- `notebooks/pipeline_tetris.ipynb` (rotation / mirror discrimination)
- `notebooks/pipeline_maze.ipynb` (maze generation + start/goal semantics)
- `notebooks/pipeline_3d.ipynb` (Ganis-Kievit 3D blocks pairs in `data/test_balanced.npy`)

This notebook calls the OpenAI API (vision).

Shared baseline generation/scoring lives in `utils/llm_baselines.py`.

Prereqs:
- `pip install openai pillow numpy opencv-python tqdm`
- Set `OPENAI_API_KEY` in your environment.


In [ ]:
from __future__ import annotations

import os
import random
import sys
from pathlib import Path

import numpy as np


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "benchmarks").exists() and (p / "notebooks").exists():
            return p
    return start


REPO_ROOT = find_repo_root(Path.cwd())
print("Repo root:", REPO_ROOT)

sys.path.insert(0, str(REPO_ROOT))

from utils.llm_baselines import (  # noqa: E402
    DEFAULT_IMG_SIZE,
    DEFAULT_MAZE_CELLS,
    DEFAULT_PAIR_PAD,
    DEFAULT_UPSCALE,
    JsonlCache,
    build_3d_blocks_samples,
    build_colored_shapes_samples,
    build_maze_solve_instances,
    build_maze_trace_samples,
    build_tetris_samples,
    cache_key,
    eval_maze_solve,
    eval_maze_trace,
    eval_rotation,
    pil_to_png_bytes,
)

# --- Config (cost/latency controls) ---
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
SEED = 0

# Match the pipeline defaults
IMG_SIZE = DEFAULT_IMG_SIZE
MAZE_CELLS = DEFAULT_MAZE_CELLS

# Make images easier for the VLM without changing the underlying grid.
UPSCALE = DEFAULT_UPSCALE  # 64 -> 256
PAIR_PAD = DEFAULT_PAIR_PAD

# Sample sizes
N_TETRIS = 500
N_COLORED_SHAPES = 500
N_3D_BLOCKS = 500
N_MAZE_TRACE = 500
N_MAZE_SOLVE = 500

# Cache LLM calls to avoid re-paying for reruns. (Historical filename retained.)
CACHE_PATH = REPO_ROOT / "benchmarks" / "llm_baselines_cache.jsonl"

random.seed(SEED)
np.random.seed(SEED)


In [ ]:
import base64
import time
from typing import Any, Optional

from PIL import Image

# --- OpenAI client + on-disk cache ---

try:
    from openai import OpenAI
except Exception as e:
    raise RuntimeError("Missing dependency: openai. Install with `pip install openai`.") from e

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY in your environment before running.")

client = OpenAI()
cache = JsonlCache(CACHE_PATH)
print(f"Loaded cache: {len(cache)} entries from {CACHE_PATH}")


def _extract_output_text(resp: Any) -> str:
    # OpenAI Python SDK (1.x) provides `output_text` on Responses API objects.
    t = getattr(resp, "output_text", None)
    if isinstance(t, str) and t.strip():
        return t.strip()

    # Chat Completions fallback shape
    try:
        return resp.choices[0].message.content.strip()  # type: ignore[attr-defined]
    except Exception:
        pass

    return str(resp).strip()


def llm_vision(
    prompt: str,
    image: Image.Image,
    *,
    model: str = MODEL,
    max_output_tokens: int = 64,
    temperature: float = 0.0,
    use_cache: bool = True,
    retries: int = 5,
) -> str:
    img = image.convert("RGB")
    png = pil_to_png_bytes(img)
    key = cache_key(
        model=model,
        prompt=prompt,
        image_bytes=png,
        max_output_tokens=max_output_tokens,
        temperature=temperature,
    )

    if use_cache:
        cached = cache.get_response(key)
        if cached is not None:
            return cached

    data_url = "data:image/png;base64," + base64.b64encode(png).decode("utf-8")

    last_err: Optional[Exception] = None
    for attempt in range(retries + 1):
        try:
            # Preferred: Responses API
            resp = client.responses.create(
                model=model,
                input=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "input_text", "text": prompt},
                            {"type": "input_image", "image_url": data_url},
                        ],
                    }
                ],
                temperature=temperature,
                max_output_tokens=max_output_tokens,
            )
            text = _extract_output_text(resp)
            break
        except Exception as e:
            last_err = e
            # Fallback: Chat Completions (older call path)
            try:
                resp = client.chat.completions.create(
                    model=model,
                    messages=[
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": prompt},
                                {"type": "image_url", "image_url": {"url": data_url}},
                            ],
                        }
                    ],
                    temperature=temperature,
                    max_tokens=max_output_tokens,
                )
                text = _extract_output_text(resp)
                break
            except Exception as e2:
                last_err = e2
                if attempt >= retries:
                    raise
                time.sleep(2**attempt)
    else:
        raise last_err or RuntimeError("Unknown OpenAI error")

    if use_cache:
        cache.put(
            key=key,
            model=model,
            prompt=prompt,
            response=text,
            meta={"max_output_tokens": max_output_tokens, "temperature": temperature},
        )

    return text


In [ ]:
# --- Build deterministic samples (for reproducibility + caching) ---

rng = random.Random(SEED)

# Rotation

tetris_samples = build_tetris_samples(
    rng,
    N_TETRIS,
    img_size=IMG_SIZE,
    upscale=UPSCALE,
    pad=PAIR_PAD,
)

colors_samples = build_colored_shapes_samples(
    rng,
    N_COLORED_SHAPES,
    img_size=IMG_SIZE,
    upscale=UPSCALE,
    pad=PAIR_PAD,
)

blocks3d_samples = build_3d_blocks_samples(
    REPO_ROOT,
    rng,
    N_3D_BLOCKS,
    upscale=UPSCALE,
    pad=PAIR_PAD,
)

# Maze

maze_trace_samples = build_maze_trace_samples(
    rng,
    N_MAZE_TRACE,
    maze_cells=MAZE_CELLS,
    upscale=UPSCALE,
)

maze_solve_instances = build_maze_solve_instances(
    rng,
    N_MAZE_SOLVE,
    maze_cells=MAZE_CELLS,
)


# --- Run ---

tetris_res = eval_rotation("tetris", tetris_samples, llm_vision=llm_vision)
colors_res = eval_rotation("colored_shapes", colors_samples, llm_vision=llm_vision)
blocks3d_res = eval_rotation("3d_blocks", blocks3d_samples, llm_vision=llm_vision)

maze_trace_res = eval_maze_trace(maze_trace_samples, llm_vision=llm_vision)
maze_solve_res = eval_maze_solve(maze_solve_instances, llm_vision=llm_vision, upscale=UPSCALE)


def pct(x: float) -> str:
    return f"{100.0 * x:.2f}%"


print("\n=== Summary (zero-shot) ===")
print(f"Rotation | tetris        : {pct(tetris_res['accuracy'])} (n={tetris_res['n']})")
print(f"Rotation | colored_shapes: {pct(colors_res['accuracy'])} (n={colors_res['n']})")
print(f"Rotation | 3d_blocks     : {pct(blocks3d_res['accuracy'])} (n={blocks3d_res['n']})")
print(f"Maze     | trace valid   : {pct(maze_trace_res['accuracy'])} (n={maze_trace_res['n']})")
print(f"Maze     | solve success : {pct(maze_solve_res['success_rate'])} (n={maze_solve_res['n']})")
